# PFΔ Task 1.1 — TPGT on Colab

**Workflow (every session):**
1. **Pull latest** — GPU runtime; run once after reconnect or when we push GitHub updates.
2. **Train** — same session; does **not** run `git pull` (change knobs at the top of that cell only).

Data: `My Drive/Foundation Model/vendor/pfdelta_task1.1`. Checkpoints: `Foundation Model/tpgt_task11_<ARCH>` (e.g. `tpgt_task11_v9`).


## 1. Pull latest

Clone or `git fetch` + `reset --hard origin/main`, mount Drive, install missing pip packages. Prints the commit hash — trust that over stale notebook markdown in the UI.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    sys.path.insert(0, str(REPO))
    from colab_bootstrap import install_required_packages, mount_drive_if_needed, pull_latest

    pull_latest(clone_if_missing=False)
    mount_drive_if_needed()
    install_required_packages()
else:
    here = Path.cwd()
    if (here / "colab_bootstrap.py").is_file():
        sys.path.insert(0, str(here))
    elif (here / "genco" / "colab_bootstrap.py").is_file():
        os.chdir(here / "genco")
        sys.path.insert(0, str(Path.cwd()))
    from colab_bootstrap import install_required_packages, pull_latest

    pull_latest(clone_if_missing=False)
    install_required_packages()


## 2. Train

Feasible **N** training (Task **1.1** by default). Set **`ARCH`** (`v7` / `v8` / `v9`), **`DATA_FRACTION`** (1.0 = full data), **`BATCH_SIZE`** (64; use 32 if OOM). Other PFΔ tasks: set **`TASK`** to `1.2`, `1.3`, `2.3`, `4.1`, …

Resumes from Drive `last.pt` when the checkpoint version matches. Changing **`DATA_FRACTION`** or arch purges stale split caches (after pull that includes the cache fix).


In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import Image, display

_repo = Path("/content/genco-figure4-task11")
if not (_repo / "colab_bootstrap.py").is_file():
    _here = Path.cwd()
    _repo = _here if (_here / "colab_bootstrap.py").is_file() else _here / "genco"
sys.path.insert(0, str(_repo))
from colab_bootstrap import (
    configure_dataset_env,
    ensure_training_cwd,
    git_head,
    mount_drive_if_needed,
    run_dir_name,
)

ensure_training_cwd()
print(f"Training from {git_head()}", flush=True)

# --- knobs ---
TASK = "1.1"
ARCH = "v9"
DATA_FRACTION = 1.0
BATCH_SIZE = 64

drive = mount_drive_if_needed()
data_id = configure_dataset_env(TASK, drive)
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(
    f"TASK={TASK} ARCH={ARCH} DATA_FRACTION={DATA_FRACTION} BATCH_SIZE={BATCH_SIZE}",
    flush=True,
)

if TASK == "1.1":
    get_ipython().run_line_magic("run", "train_tpgt_task11.py")
else:
    get_ipython().run_line_magic("run", f"train_tpgt_task11.py --task {TASK}")

run = run_dir_name(TASK, ARCH)
foundation = drive / "Foundation Model"
if TASK == "1.1":
    bar = foundation / "results" / run / f"figure4_task{data_id}_tpgt.png"
    maps_root = foundation / run / "bus_error_maps"
else:
    bar = foundation / run / "results" / f"figure4_task{data_id}_tpgt.png"
    maps_root = foundation / run / "bus_error_maps"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(maps_root.glob("seed_*/epoch_*")) if maps_root.is_dir() else []
if maps:
    latest = maps[-1]
    print(f"Latest bus error maps: {latest}", flush=True)
    for png in sorted(latest.glob("*.png"))[:6]:
        display(Image(filename=str(png)))
